# 01 - Database Schema Design & Market Data Ingestion

**Project:** SEC Event Study Analytics System  
**Author contribution:** Full pipeline design — PostgreSQL schema, CIK-Ticker mapping, Yahoo Finance ingestion via yfinance

## What this notebook does
1. Creates PostgreSQL schema: `company` and `daily_price` tables
2. Loads 927 tech sector companies from SEC `company_tickers.json`
3. Ingests 10 years of daily OHLCV price data from Yahoo Finance for all companies

## Setup
```bash
pip install -r requirements.txt
# Copy .env.example to .env and fill in your DB credentials
```

## Key design decisions
- `company` table uses `company_id` (SERIAL) as PK for join stability; `cik` stored as zero-padded 10-digit string
- `daily_price` uses composite PK `(company_id, trade_date)` for efficient time-series queries
- `ON CONFLICT DO NOTHING` on insert allows safe re-runs without duplicates
- Date range is parameterized: change `FIXED_START` to adjust the ingestion window


# Connect Database

In [4]:
import os
import psycopg
conn = psycopg.connect(
    host=os.environ.get("PG_HOST", "localhost"),
    port=os.environ.get("PG_PORT", "5432"),
    dbname=os.environ.get("PG_DBNAME", "final"), 
    user=os.environ.get("PG_USER", "postgres"),
    password=os.environ.get("PG_PASSWORD", "your_password") 
)

In [ ]:
cur = conn.cursor()
print("Connected to PostgreSQL")

cur.execute("""
    DROP TABLE IF EXISTS daily_price;
    DROP TABLE IF EXISTS company;
""")

# Create Tech Company Table

In [3]:
cur.execute("""
    CREATE TABLE company (
        company_id      SERIAL PRIMARY KEY, 
        cik             VARCHAR(10) NOT NULL UNIQUE,
        ticker          VARCHAR(10) NOT NULL UNIQUE,
        company_name    TEXT NOT NULL,
        sic             INTEGER,
        sic_description TEXT
    );""")
conn.commit()
print("Table 'company' created.")

Table 'company' created.


In [4]:
import pandas as pd
df = pd.read_csv("primary_ticker_only.csv")
df.head()

df_clean = df.rename(columns={
    "ticker": "ticker",
    "title": "company_name",
    "CIK": "cik",
    "SIC": "sic",
    "SIC_description": "sic_description"
})

# Convert CIK to 10 digits
df_clean["cik"] = df_clean["cik"].astype(str).str.zfill(10)
df_clean.head()

,ticker,company_name,cik,sic,sic_description
0,BKTI,BK Technologies Corp,0000002186,3663,Radio & Tv Broadcasting & Communications Equip...
1,AMD,ADVANCED MICRO DEVICES INC,0000002488,3674,Semiconductors & Related Devices
2,SWKS,"SKYWORKS SOLUTIONS, INC.",0000004127,3674,Semiconductors & Related Devices
3,ADI,ANALOG DEVICES INC,0000006281,3674,Semiconductors & Related Devices
4,AMAT,APPLIED MATERIALS INC /DE,0000006951,3674,Semiconductors & Related Devices


In [5]:
for _, row in df_clean.iterrows():
    cur.execute("""
        INSERT INTO company (cik, ticker, company_name, sic, sic_description)
        VALUES (%s, %s, %s, %s, %s);
    """, (
        row["cik"],
        row["ticker"],
        row["company_name"],
        int(row["sic"]) if not pd.isna(row["sic"]) else None,
        row["sic_description"]
    ))

conn.commit()
print("All companies inserted!")

All companies inserted!


In [7]:
cur.execute("SELECT company_id, cik, ticker, company_name FROM company LIMIT 10;")
for row in cur.fetchall():
    print(row)
cur.execute("SELECT COUNT(*) FROM company;")
print("Total companies:", cur.fetchone()[0])

cur.execute("""
    SELECT
        COUNT(*) FILTER (WHERE cik IS NULL)              AS cik_nulls,
        COUNT(*) FILTER (WHERE ticker IS NULL)           AS ticker_nulls,
        COUNT(*) FILTER (WHERE company_name IS NULL)     AS company_name_nulls,
        COUNT(*) FILTER (WHERE sic IS NULL)              AS sic_nulls,
        COUNT(*) FILTER (WHERE sic_description IS NULL)  AS sic_description_nulls
    FROM company;
""")

print("NULL count per column:", cur.fetchone())

(1, '0000002186', 'BKTI', 'BK Technologies Corp')
(2, '0000002488', 'AMD', 'ADVANCED MICRO DEVICES INC')
(3, '0000004127', 'SWKS', 'SKYWORKS SOLUTIONS, INC.')
(4, '0000006281', 'ADI', 'ANALOG DEVICES INC')
(5, '0000006951', 'AMAT', 'APPLIED MATERIALS INC /DE')
(6, '0000008146', 'ALOT', 'AstroNova, Inc.')
(7, '0000008670', 'ADP', 'AUTOMATIC DATA PROCESSING INC')
(8, '0000016058', 'CACI', 'CACI INTERNATIONAL INC /DE/')
(9, '0000023197', 'CMTL', 'COMTECH TELECOMMUNICATIONS CORP /DE/')
(10, '0000026058', 'CTS', 'CTS CORP')
Total companies: 927
NULL count per column: (0, 0, 0, 0, 0)


# Create daily_price table & insert data from yahoo finance

In [1]:
!pip install -U "psycopg[binary]" yfinance pandas


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import psycopg

conn = psycopg.connect(
    host=os.environ.get("PG_HOST", "localhost"),
    port=os.environ.get("PG_PORT", "5432"),
    dbname=os.environ.get("PG_DBNAME", "final"),
    user=os.environ.get("PG_USER", "postgres"),
    password=os.environ.get("PG_PASSWORD", "your_password")
)
cur = conn.cursor()
print("Connected to Final database.")

Connected to Final database.


In [4]:
cur.execute("""
    DROP TABLE IF EXISTS daily_price;

    CREATE TABLE daily_price (
        company_id  INTEGER NOT NULL REFERENCES company(company_id),
        trade_date  DATE    NOT NULL,
        open        NUMERIC(18,4),
        high        NUMERIC(18,4),
        low         NUMERIC(18,4),
        close       NUMERIC(18,4),
        adj_close   NUMERIC(18,4),
        volume      BIGINT,
        PRIMARY KEY (company_id, trade_date)
    );
""")
conn.commit()

print("Table 'daily_price' created.")

Table 'daily_price' created.


In [5]:
import pandas as pd

df_comp = pd.read_sql("SELECT company_id, ticker FROM company ORDER BY company_id;", conn)

print("Total companies:", len(df_comp))
df_comp.head()

Total companies: 927


C:\Users\Milan_Chen\AppData\Local\Temp\ipykernel_53728\587111543.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_comp = pd.read_sql("SELECT company_id, ticker FROM company ORDER BY company_id;", conn)


,company_id,ticker
0,1,BKTI
1,2,AMD
2,3,SWKS
3,4,ADI
4,5,AMAT


Set the starting point as November 28, 2015 and the ending point as today. Daily OHLCV of grabbing a certain ticker from FIXED_START to TODAY. Take as much as you have. If the company has been listed for less than ten years, the returned data will automatically decrease.

In [11]:
import yfinance as yf
from datetime import datetime

FIXED_START = "2015-11-28"                       
# fixed beginning date
TODAY = datetime.today().strftime("%Y-%m-%d")    
# end date = today

print("Fetching window:", FIXED_START, "→", TODAY)

def fetch_price_fixed_window(ticker: str):

    try:
        df = yf.download(
            ticker,
            start=FIXED_START,
            end=TODAY,
            auto_adjust=False,
            progress=False
        )

        if df.empty:
            print(f"⚠️ {ticker}: no price data found.")
            return None

        df.reset_index(inplace=True)

        df.rename(columns={
            "Date": "trade_date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume"
        }, inplace=True)

        df = df[["trade_date", "open", "high", "low", "close", "adj_close", "volume"]]

        return df

    except Exception as e:
        print(f"❌ Error downloading {ticker}: {e}")
        return None

Fetching window: 2015-11-28 → 2025-11-28


rec is a tuple, corresponding to the column order:  
(trade_date, open, high, low, close, adj_close, volume)

In [12]:
import math

for _, row in df_comp.iterrows():
    cid = int(row["company_id"])
    ticker = row["ticker"]

    print(f"\nFetching from {FIXED_START} to {TODAY} for {ticker} (company_id={cid}) ...")

    df_price = fetch_price_fixed_window(ticker)
    if df_price is None:
        continue

    print("  Columns from yfinance:", list(df_price.columns))

    for rec in df_price.itertuples(index=False, name=None):
        
        trade_date_raw, open_raw, high_raw, low_raw, close_raw, adj_close_raw, vol_raw = rec

        if isinstance(trade_date_raw, pd.Timestamp):
            trade_date = trade_date_raw.date()
        else:
            trade_date = trade_date_raw

        def to_num(x):
            if x is None:
                return None
            if isinstance(x, float) and math.isnan(x):
                return None
            return float(x)

        open_      = to_num(open_raw)
        high_      = to_num(high_raw)
        low_       = to_num(low_raw)
        close_     = to_num(close_raw)
        adj_close_ = to_num(adj_close_raw)

        vol = vol_raw
        if vol is None or (isinstance(vol, float) and math.isnan(vol)):
            vol = None
        else:
            vol = int(vol)

        cur.execute("""
            INSERT INTO daily_price
                (company_id, trade_date, open, high, low, close, adj_close, volume)
            VALUES
                (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (company_id, trade_date) DO NOTHING;
        """, (
            cid,
            trade_date,
            open_,
            high_,
            low_,
            close_,
            adj_close_,
            vol
        ))

    conn.commit()
    print(f"✔ Inserted {len(df_price)} rows for {ticker}.")


⬇️ Fetching from 2015-11-28 to 2025-11-28 for BKTI (company_id=1) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'BKTI'), ('high', 'BKTI'), ('low', 'BKTI'), ('close', 'BKTI'), ('adj_close', 'BKTI'), ('volume', 'BKTI')]
✔ Inserted 2514 rows for BKTI.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for AMD (company_id=2) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'AMD'), ('high', 'AMD'), ('low', 'AMD'), ('close', 'AMD'), ('adj_close', 'AMD'), ('volume', 'AMD')]
✔ Inserted 2514 rows for AMD.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for SWKS (company_id=3) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'SWKS'), ('high', 'SWKS'), ('low', 'SWKS'), ('close', 'SWKS'), ('adj_close', 'SWKS'), ('volume', 'SWKS')]
✔ Inserted 2514 rows for SWKS.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for ADI (company_id=4) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'ADI'), ('high', 'ADI'), ('low', 'ADI'), ('close', 'ADI'), ('adj_close', 'ADI'), ('volume', 'A


1 Failed download:
['CAST']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-28 -> 2025-11-28)')


⚠️ CAST: no price data found.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for LITE (company_id=506) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'LITE'), ('high', 'LITE'), ('low', 'LITE'), ('close', 'LITE'), ('adj_close', 'LITE'), ('volume', 'LITE')]
✔ Inserted 2514 rows for LITE.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for FLUT (company_id=507) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'FLUT'), ('high', 'FLUT'), ('low', 'FLUT'), ('close', 'FLUT'), ('adj_close', 'FLUT'), ('volume', 'FLUT')]
✔ Inserted 2514 rows for FLUT.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for HCAT (company_id=508) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'HCAT'), ('high', 'HCAT'), ('low', 'HCAT'), ('close', 'HCAT'), ('adj_close', 'HCAT'), ('volume', 'HCAT')]
✔ Inserted 1596 rows for HCAT.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for ZSPC (company_id=509) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'ZSPC'), ('high', 'ZSPC'), ('low', 'ZSPC'), ('c


1 Failed download:
['NTCS']: YFTzMissingError('possibly delisted; no timezone found')


⚠️ NTCS: no price data found.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for AIXI (company_id=833) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'AIXI'), ('high', 'AIXI'), ('low', 'AIXI'), ('close', 'AIXI'), ('adj_close', 'AIXI'), ('volume', 'AIXI')]
✔ Inserted 684 rows for AIXI.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for PODC (company_id=834) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'PODC'), ('high', 'PODC'), ('low', 'PODC'), ('close', 'PODC'), ('adj_close', 'PODC'), ('volume', 'PODC')]
✔ Inserted 558 rows for PODC.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for MFI (company_id=835) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'MFI'), ('high', 'MFI'), ('low', 'MFI'), ('close', 'MFI'), ('adj_close', 'MFI'), ('volume', 'MFI')]
✔ Inserted 403 rows for MFI.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for TANAF (company_id=836) ...
  Columns from yfinance: [('trade_date', ''), ('open', 'TANAF'), ('high', 'TANAF'), ('low', 'TANAF'), ('close', 

...

⚠️ EVON: no price data found.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for BRAI (company_id=925) ...
⚠️ BRAI: no price data found.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for ZSSK (company_id=926) ...



1 Failed download:
['ZSSK']: YFTzMissingError('possibly delisted; no timezone found')


⚠️ ZSSK: no price data found.

⬇️ Fetching from 2015-11-28 to 2025-11-28 for XYJG (company_id=927) ...



1 Failed download:
['XYJG']: YFTzMissingError('possibly delisted; no timezone found')


⚠️ XYJG: no price data found.


In [4]:
cur.execute("""
    SELECT company_id, trade_date, open, close
    FROM daily_price
    ORDER BY trade_date DESC
    LIMIT 10;
""")
rows = cur.fetchall()

print("\nSample rows from daily_price:")
for r in rows:
    print(r)


Sample rows from daily_price:
(10, datetime.date(2025, 11, 26), Decimal('42.2400'), Decimal('42.5800'))
(15, datetime.date(2025, 11, 26), Decimal('24.1000'), Decimal('23.9800'))
(8, datetime.date(2025, 11, 26), Decimal('616.6700'), Decimal('615.3500'))
(9, datetime.date(2025, 11, 26), Decimal('3.1400'), Decimal('3.0300'))
(1, datetime.date(2025, 11, 26), Decimal('66.4200'), Decimal('63.9700'))
(3, datetime.date(2025, 11, 26), Decimal('63.6000'), Decimal('65.3400'))
(2, datetime.date(2025, 11, 26), Decimal('210.0500'), Decimal('214.2400'))
(6, datetime.date(2025, 11, 26), Decimal('8.0000'), Decimal('7.8600'))
(4, datetime.date(2025, 11, 26), Decimal('255.2200'), Decimal('257.9200'))
(18, datetime.date(2025, 11, 26), Decimal('12.9900'), Decimal('13.4000'))


Check the data download status

In [5]:
print("\n" + "="*50)
print("Data download status check:")
print("="*50)

cur.execute("""
    SELECT COUNT(DISTINCT company_id) 
    FROM daily_price;
""")
companies_with_data = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM company;")
total_companies = cur.fetchone()[0]

companies_without_data = total_companies - companies_with_data

print(f"Number of total comapnies: {total_companies}")
print(f"Number of companies with data: {companies_with_data}")
print(f"Number of companies without data: {companies_without_data}")
print(f"Data coverage rate: {companies_with_data/total_companies*100:.1f}%")

if companies_without_data > 0:
    print(f"\nThe list of companies without data:")
    cur.execute("""
        SELECT c.company_id, c.ticker, c.company_name
        FROM company c
        LEFT JOIN daily_price dp ON c.company_id = dp.company_id
        WHERE dp.company_id IS NULL
        ORDER BY c.ticker;
    """)
    
    no_data_companies = cur.fetchall()
    for company_id, ticker, name in no_data_companies[:10]:
        print(f"  {ticker}: {name}")
    
    if len(no_data_companies) > 10:
        print(f"  ... There are {len(no_data_companies)-10}  more companies")

print(f"\nData volume statistics:")
cur.execute("""
    SELECT 
        COUNT(DISTINCT company_id) as companies,
        MIN(cnt) as min_records,
        MAX(cnt) as max_records,
        AVG(cnt)::INT as avg_records
    FROM (
        SELECT company_id, COUNT(*) as cnt
        FROM daily_price 
        GROUP BY company_id
    ) t;
""")

stats = cur.fetchone()
print(f"The number of companies with data: {stats[0]}")
print(f"Minimum number of records: {stats[1]}")
print(f"Maximum number of records: {stats[2]}")  
print(f"Average number of records: {stats[3]}")


Data download status check:
Number of total comapnies: 927
Number of companies with data: 907
Number of companies without data: 20
Data coverage rate: 97.8%

The list of companies without data:
  ALD: Altech Digital Co., Ltd.
  ALXY: MADE IN USA INC.
  BAO: BAO Holding Ltd.
  BRAI: Braiin Ltd
  CAST: FreeCast, Inc.
  DBIM: Dbim Holdings Ltd
  ECST: ECST Holdings Ltd
  EVON: EvoNexus Group LTD
  GBH: Gigabit Inc.
  GDNT: Guident Corp.
  ... There are 10  more companies

Data volume statistics:
The number of companies with data: 907
Minimum number of records: 2
Maximum number of records: 2514
Average number of records: 1770
